# Desafio 1: chegaram dados novos

**Contexto.** É janeiro de 2013. Chegaram os dados do 2º semestre de 2012. Missão: versionar os dados (`dados-v3`),
retreinar com o pipeline e **decidir, com evidência**, se o modelo novo substitui o `@champion`.


In [ ]:
import os, sys
from pathlib import Path

# Trabalha a partir da raiz do repositório (a pasta que contém o params.yaml)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "params.yaml").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

## Passo 1 · Dados v3 (a receita da T4)

Os mesmos 4 passos: `dvc add`, `git add` + `commit`, `git tag`, `dvc push`.

In [ ]:
%%bash

#python src/get_data.py --until 2012-12-31
#dvc add data/raw/bike.csv
#git add data/raw/bike.csv.dvc
#git commit -m "Dados v3: 2012 completo"
#git tag dados-v3
#dvc push

# Conferência
git tag
git log --oneline --decorate -- data/raw/bike.csv.dvc

## Passo 2 · O `compare_models.py` está completo?


In [ ]:
%%bash
grep -n "TODO" -A1 desafio01/compare_models.py

## Passo 3 · O estágio `compare` no `dvc.yaml`


In [ ]:
%%bash
tail -n 12 dvc.yaml
dvc dag | cat

## Passo 4 · Retreinar e comparar



In [ ]:
%%bash
dvc repro

In [ ]:
%%bash
cat reports/comparacao.json
echo
dvc metrics diff

In [ ]:
%%bash
dvc push

## Passo 5 · Decidir



In [ ]:
%%bash
#python desafio01/compare_models.py --promover

### Conferir o Registry (só leitura, pode rodar sempre)


In [ ]:
import mlflow
from mlflow import MlflowClient
from bike import MODEL_NAME, tracking_uri

mlflow.set_tracking_uri(tracking_uri())
client = MlflowClient()

champ = client.get_model_version_by_alias(MODEL_NAME, "champion")
print(f"@champion -> versão {champ.version}")
for k, v in sorted(champ.tags.items()):
    print(f"  {k}: {v}")

## Passo 6 · Dados para o relatório


In [ ]:
%%bash
for tag in dados-v1 dados-v2 dados-v3; do
  echo "$tag $(git show $tag:data/raw/bike.csv.dvc | awk '/md5/{print $3}')"
done

In [ ]:
import json
r = json.load(open("reports/comparacao.json"))
t = r["teste"]
print(f"Período do teste: {t['inicio']} a {t['fim']} ({t['linhas']} linhas)\n")
print(f"{'alias':<12}{'versão':>7}{'RMSE':>9}{'MAE':>9}{'R²':>9}")
for alias in ["champion", "challenger"]:
    m = r[alias]
    print(f"{alias:<12}{m['versao']:>7}{m['rmse']:>9.2f}{m['mae']:>9.2f}{m['r2']:>9.3f}")

## Passo 7 · Entrega


In [ ]:
%%bash
#git add dvc.yaml dvc.lock metrics.json reports/ desafio01/
#git commit -m "Desafio 1: dados v3, estágio compare e decisão"
#git push
#git push --tags
git status --short